In [0]:
# Databricks notebook source

# COMMAND ----------

# MAGIC %md
# MAGIC # Gold Market Feature Exploration
# MAGIC
# MAGIC This notebook:
# MAGIC
# MAGIC 1. Imports shared modeling configuration from `src/modeling`.
# MAGIC 2. Loads the Gold Kalshi CFB market-behavior table.
# MAGIC 3. Validates feature completeness and eligibility.
# MAGIC 4. Examines distributions and correlations.
# MAGIC 5. Prepares a modeling dataset for Isolation Forest.
# MAGIC
# MAGIC Future-price columns are used only to evaluate whether detected
# MAGIC anomalies subsequently mean-revert.

In [0]:
# Add the repository root to Python’s import path

import sys
from pathlib import Path


def find_repository_root(start_path: Path) -> Path:
    """
    Walk upward until the repository root containing src/modeling is found.
    """
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "src" / "modeling").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find repository root containing src/modeling. "
        f"Notebook working directory: {start_path}"
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

print(f"Repository root: {REPOSITORY_ROOT}")

In [0]:
# Import shared configuration

from src.modeling.model_config import (
    GOLD_TABLE,
    IDENTIFIER_COLUMNS,
    FEATURE_COLUMNS,
    EVALUATION_COLUMNS,
    ELIGIBILITY_COLUMN,
    ISOLATION_FOREST_PARAMETERS,
)

print(f"Gold table: {GOLD_TABLE}")
print(f"Feature count: {len(FEATURE_COLUMNS)}")
print(f"Features: {FEATURE_COLUMNS}")

In [0]:
# Load the Gold table

from pyspark.sql import functions as F


required_columns = list(
    dict.fromkeys(
        IDENTIFIER_COLUMNS
        + FEATURE_COLUMNS
        + EVALUATION_COLUMNS
        + [
            ELIGIBILITY_COLUMN,
            "is_trade_eligible",
            "selection_team",
            "opponent_team",
            "market_side",
            "game_start_utc",
        ]
    )
)

gold_df = (
    spark.read.table(GOLD_TABLE)
    .select(*required_columns)
)

print(f"Loaded table: {GOLD_TABLE}")
print(f"Selected columns: {len(gold_df.columns)}")

display(gold_df.limit(20))

In [0]:
# Validate the expected schema

available_columns = set(
    spark.table(GOLD_TABLE).columns
)

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in available_columns
]

if missing_columns:
    raise ValueError(
        "The Gold table is missing required columns: "
        f"{missing_columns}"
    )

print("Gold table contains all required modeling columns.")

In [0]:
# Basic table profile

profile_df = gold_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("kalshi_ticker").alias("ticker_count"),
    F.countDistinct("espn_event_id").alias("game_count"),
    F.min("minute_ts").alias("minimum_minute_ts"),
    F.max("minute_ts").alias("maximum_minute_ts"),
    F.sum(
        F.col(ELIGIBILITY_COLUMN).cast("long")
    ).alias("training_eligible_rows"),
    F.sum(
        F.col("is_trade_eligible").cast("long")
    ).alias("trade_eligible_rows"),
)

display(profile_df)

In [0]:
# Check null rates

columns_to_profile = FEATURE_COLUMNS + EVALUATION_COLUMNS

null_rate_expressions = []

for column_name in columns_to_profile:
    null_rate_expressions.append(
        F.mean(
            F.col(column_name).isNull().cast("double")
        ).alias(f"{column_name}_null_rate")
    )

null_rates_df = gold_df.agg(*null_rate_expressions)

display(null_rates_df)

In [0]:
# Restrict to eligible historical observations

eligible_df = (
    gold_df
    .filter(F.col(ELIGIBILITY_COLUMN))
    .dropna(
        subset=FEATURE_COLUMNS
    )
)

eligible_profile_df = eligible_df.agg(
    F.count("*").alias("eligible_complete_rows"),
    F.countDistinct("kalshi_ticker").alias("ticker_count"),
    F.countDistinct("espn_event_id").alias("game_count"),
)

display(eligible_profile_df)

In [0]:
# Descriptive feature statistics

display(
    eligible_df.select(*FEATURE_COLUMNS).summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
    )
)

In [0]:
# Distribution by game phase

game_phase_df = (
    eligible_df
    .withColumn(
        "game_phase",
        F.when(
            F.col("minutes_from_game_start") < 30,
            F.lit("15-29 minutes"),
        )
        .when(
            F.col("minutes_from_game_start") < 60,
            F.lit("30-59 minutes"),
        )
        .when(
            F.col("minutes_from_game_start") < 90,
            F.lit("60-89 minutes"),
        )
        .otherwise(
            F.lit("90-105 minutes")
        ),
    )
    .groupBy("game_phase")
    .agg(
        F.count("*").alias("observation_count"),
        F.avg("current_price").alias("average_price"),
        F.avg("relative_volume_5m").alias(
            "average_relative_volume_5m"
        ),
        F.avg("price_volatility_10m").alias(
            "average_price_volatility_10m"
        ),
        F.avg("future_price_change_5m").alias(
            "average_future_price_change_5m"
        ),
    )
    .orderBy("game_phase")
)

display(game_phase_df)

In [0]:
# Convert the modeling sample to Pandas

MAX_MODELING_ROWS = 250_000

eligible_row_count = eligible_df.count()

sample_fraction = min(
    1.0,
    MAX_MODELING_ROWS / max(eligible_row_count, 1),
)

print(f"Eligible rows: {eligible_row_count:,}")
print(f"Sample fraction: {sample_fraction:.4f}")

modeling_sample_df = (
    eligible_df
    .sample(
        withReplacement=False,
        fraction=sample_fraction,
        seed=42,
    )
    .select(
        *IDENTIFIER_COLUMNS,
        *FEATURE_COLUMNS,
        *EVALUATION_COLUMNS,
    )
)

modeling_pdf = modeling_sample_df.toPandas()

print(f"Pandas modeling rows: {len(modeling_pdf):,}")

In [0]:
# Prepare the feature matrix

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


X_raw = modeling_pdf[FEATURE_COLUMNS].copy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X_raw)
X_scaled = scaler.fit_transform(X_imputed)

print(f"Feature matrix shape: {X_scaled.shape}")

feature_summary_pdf = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "mean_after_scaling": X_scaled.mean(axis=0),
        "std_after_scaling": X_scaled.std(axis=0),
    }
)

display(feature_summary_pdf)

In [0]:
# Fit the initial Isolation Forest

from sklearn.ensemble import IsolationForest


isolation_forest = IsolationForest(
    **ISOLATION_FOREST_PARAMETERS
)

isolation_forest.fit(X_scaled)

# sklearn returns lower decision values for more abnormal observations.
modeling_pdf["isolation_decision_score"] = (
    isolation_forest.decision_function(X_scaled)
)

# Convert it so larger values mean "more anomalous."
modeling_pdf["anomaly_score"] = (
    -modeling_pdf["isolation_decision_score"]
)

modeling_pdf["is_anomaly_default"] = (
    isolation_forest.predict(X_scaled) == -1
)

display(
    modeling_pdf[
        IDENTIFIER_COLUMNS
        + FEATURE_COLUMNS
        + [
            "anomaly_score",
            "is_anomaly_default",
            "future_price_change_5m",
        ]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False,
    )
    .head(100)
)

In [0]:
# Define anomaly severity by percentile

ANOMALY_PERCENTILE = 0.99

anomaly_threshold = modeling_pdf[
    "anomaly_score"
].quantile(ANOMALY_PERCENTILE)

modeling_pdf["is_top_anomaly"] = (
    modeling_pdf["anomaly_score"] >= anomaly_threshold
)

print(f"Anomaly percentile: {ANOMALY_PERCENTILE:.1%}")
print(f"Anomaly threshold: {anomaly_threshold:.6f}")
print(
    "Top anomaly rows: "
    f"{modeling_pdf['is_top_anomaly'].sum():,}"
)

In [0]:
# Evaluate mean reversion

modeling_pdf["shock_direction"] = np.sign(
    modeling_pdf["price_change_5m"]
)

modeling_pdf["mean_reversion_return_5m"] = (
    -modeling_pdf["shock_direction"]
    * modeling_pdf["future_price_change_5m"]
)

modeling_pdf["did_mean_revert_5m"] = (
    modeling_pdf["mean_reversion_return_5m"] > 0
)

evaluation_pdf = (
    modeling_pdf
    .groupby("is_top_anomaly", dropna=False)
    .agg(
        observation_count=(
            "kalshi_ticker",
            "size",
        ),
        mean_anomaly_score=(
            "anomaly_score",
            "mean",
        ),
        mean_future_price_change_5m=(
            "future_price_change_5m",
            "mean",
        ),
        mean_reversion_rate_5m=(
            "did_mean_revert_5m",
            "mean",
        ),
        mean_reversion_return_5m=(
            "mean_reversion_return_5m",
            "mean",
        ),
    )
    .reset_index()
)

display(evaluation_pdf)

In [0]:
# Evaluate multiple anomaly buckets

modeling_pdf["anomaly_percentile_bucket"] = pd.qcut(
    modeling_pdf["anomaly_score"],
    q=10,
    labels=[
        "00-10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "90-100%",
    ],
    duplicates="drop",
)

bucket_evaluation_pdf = (
    modeling_pdf
    .groupby(
        "anomaly_percentile_bucket",
        observed=True,
    )
    .agg(
        observation_count=(
            "kalshi_ticker",
            "size",
        ),
        average_anomaly_score=(
            "anomaly_score",
            "mean",
        ),
        mean_reversion_rate_5m=(
            "did_mean_revert_5m",
            "mean",
        ),
        average_mean_reversion_return_5m=(
            "mean_reversion_return_5m",
            "mean",
        ),
    )
    .reset_index()
)

display(bucket_evaluation_pdf)